# Milestone 4: speculative decoding upgrade (prompt-lookup drafter + cost model)
Kaggle: **GPU T4 x2, Internet ON**, Run All. About 15-20 min. Uses the final adapters from HuggingFace, so nothing is retrained.

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
TARGET_ADAPTER = 'usmanbari2005/trialmatch-qwen25-1p5b-lora-v2'
DRAFT_ADAPTER = 'usmanbari2005/trialmatch-qwen25-0p5b-draft-lora'
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
%cd /kaggle/working
!rm -rf trial-eligibility-transformer
!git clone https://github.com/UsmanBari/trial-eligibility-transformer.git
%cd /kaggle/working/trial-eligibility-transformer
!pip install -q -r requirements.txt && pip install -q -e .
!python -m pytest -q tests
!python scripts/prepare_data.py | tail -2

/kaggle/working
Cloning into 'trial-eligibility-transformer'...
remote: Enumerating objects: 243, done.
remote: Counting objects: 100% (243/243), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 243 (delta 118), reused 217 (delta 92), pack-reused 0 (from 0)
Receiving objects: 100% (243/243), 1.09 MiB | 7.55 MiB/s, done.
Resolving deltas: 100% (118/118), done.
/kaggle/working/trial-eligibility-transformer
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
litellm 1.93.2 requi

## Speculative decoding: prompt lookup vs base draft vs tuned draft, k = 2/4/6/8 (fp16), with measured cost model

In [3]:
!python scripts/bench_spec.py --adapter {TARGET_ADAPTER} --drafts lookup=prompt_lookup base=Qwen/Qwen2.5-0.5B-Instruct tuned=Qwen/Qwen2.5-0.5B-Instruct@{DRAFT_ADAPTER} --ks 2 4 6 8 --dtype float16

tokenizer_config.json: 7.30kB [00:00, 29.5MB/s]
vocab.json: 2.78MB [00:00, 44.7MB/s]
merges.txt: 1.67MB [00:00, 135MB/s]
tokenizer.json: 7.03MB [00:00, 204MB/s]
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 4.76MB/s]
I0000 00:00:1793969321.728136     284 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:07<00:00, 420MB/s]
adapter_config.json: 100%|█████████████████████| 729/729 [00:00<00:00, 7.86MB/s]
adapter_model.safetensors: 100%|███████████| 73.9M/73.9M [00:01<00:00, 46.4MB/s]
[cost] target 1-token step 30.8 ms; verify k+1 tokens: k=2: 33.7 ms (x1.09), k=4: 34.1 ms (x1.10), k=6: 34.1 ms (x1.11), k=8: 33.9 ms (x1.10)
[baseline] task       312 tokens    31.0 tok/s  latency   503.6 ms/prompt
[baseline] summary    663 tokens    32.1 tok/s  latency  3437.0 ms/prompt
[lookup k=2] task     equal 20/20  accept 0.23    36.5 tok/s  speedup x1.18  tok/target-pass 1.31  pr

## fp32 exactness check (lookup and tuned draft, k=4)

In [ ]:
!python scripts/bench_spec.py --adapter {TARGET_ADAPTER} --drafts lookup=prompt_lookup tuned=Qwen/Qwen2.5-0.5B-Instruct@{DRAFT_ADAPTER} --ks 4 --dtype float32 --n-task 10 --n-summary 3

In [ ]:
!python scripts/make_bench_figures.py
import json
d = json.load(open('results/bench/spec_float16.json'))
print('cost model:', json.dumps(d['cost_model'], indent=1))
for name, r in d['drafts'].items():
    print(name, 'c =', r.get('c'))

## Download `m4_results.zip`

In [4]:
!zip -qr /kaggle/working/m4_results.zip results/bench results/figures
!ls -la /kaggle/working

total 1260
drwxr-xr-x  4 root root    4096 Nov  6 13:01 .
drwxr-xr-x  5 root root    4096 Nov  6 12:47 ..
-rw-r--r--  1 root root 1273681 Nov  6 13:01 m4_results.zip
drwxr-xr-x 12 root root    4096 Nov  6 12:48 trial-eligibility-transformer
drwxr-xr-x  2 root root    4096 Nov  6 12:47 .virtual_documents
